In [ ]:
import sys
import os
import pandas as pd
import time
import asyncio
from tqdm.asyncio import tqdm # Tạo thanh tiến trình cho đẹp
from dotenv import load_dotenv
import google.generativeai as genai

# --- FIX PATH TOÀN DIỆN (Import + Working Directory) ---
current_dir = os.getcwd()

# 1. Xác định thư mục gốc (IT_Project_2526)
# (Logic: Đang ở evaluate/evaluate_baseline nên lùi 2 cấp là ra gốc)
project_root = os.path.abspath(os.path.join(current_dir, "../.."))

# 2. QUAN TRỌNG NHẤT: Ép Jupyter "nhảy" về thư mục gốc để đứng
# Giúp lệnh open("./server/data...") trong code nguồn hoạt động đúng
os.chdir(project_root)
print(f"📂 Current Working Directory: {os.getcwd()}")

# 3. Setup đường dẫn Import (sys.path)
server_path = os.path.join(project_root, "server")
src_path = os.path.join(server_path, "src")

if project_root not in sys.path: sys.path.insert(0, project_root)
if server_path not in sys.path: sys.path.insert(0, server_path)
if src_path not in sys.path: sys.path.insert(0, src_path)

print(f"✅ Đã nạp sys.path")

# 4. Load .env
load_dotenv(os.path.join(project_root, ".env"))

# 5. Setup Baseline Model
import google.generativeai as genai
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
baseline_model = genai.GenerativeModel("gemini-2.5-flash-lite")

In [ ]:
# Đường dẫn file csv của bạn
file_path = "evaluate/evaluate_data/Q&A.csv"

# Đọc file (Dùng encoding utf-8-sig nếu file tạo từ Excel để tránh lỗi font)
df = pd.DataFrame(pd.read_csv(file_path))
df = df[156::]

# Đảm bảo tên cột chuẩn (xóa khoảng trắng thừa nếu có)
df.columns = [c.strip() for c in df.columns]

print(f"📦 Đã load {len(df)} câu hỏi.")
display(df.head(3)) # Hiển thị 3 dòng đầu

In [ ]:
# --- IMPORT MODEL TỪ SERVER ---
# Lưu ý: Phải có 'server.' ở đầu
from server.src.core.Retrieval import Retrieval
from server.src.core.Generator import Generator
from server.src.core.ChatManager import ChatManager
from server.src.core.loader import load_components
from server.users.user_manager import load_user

import asyncio

print("⏳ Đang khởi tạo hệ thống EduRAG...")

# 1. Load User
# Vì hàm load_user là async, phải dùng await
user = await load_user("U000")

# 2. Load Components
multi_purposes_model, router_model, retriever, generator = load_components()

# 3. Khởi tạo ChatManager
chat_manager = ChatManager(user, multi_purposes_model, router_model, retriever, generator)

print("✅ EduRAG System đã sẵn sàng!")

In [ ]:
ans,_ = await chat_manager.handle_query("Nguyễn Ái Quốc là ai?", 10)
print(ans)

In [ ]:
async def get_baseline_response(question):
    """Gọi trực tiếp Google Gemini, không RAG, không context"""
    try:
        # Prompt đơn giản để model trả lời trực tiếp
        prompt = f"Trả lời câu hỏi sau một cách ngắn gọn và chính xác:\nCâu hỏi: {question}"
        
        # Gọi model (chạy trong executor để không chặn event loop)
        response = await asyncio.to_thread(baseline_model.generate_content, prompt)
        return response.text.strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

In [ ]:
ans = get_baseline_response("Nguyễn Ái Quốc là ai?")
ans

In [ ]:
import time
import pandas as pd
import os
import asyncio  # Dùng thư viện này để sleep trong môi trường async tốt hơn
from tqdm.asyncio import tqdm as async_tqdm # Dùng tqdm cho async nếu có thể, hoặc dùng tqdm thường cũng được
from tqdm import tqdm
import traceback

# --- CẤU HÌNH AN TOÀN ---
TOP_K = 20
BATCH_SIZE = 10
OUTPUT_FILE = './evaluation_results_checkpoint.csv'
SLEEP_TIME = 25      # Tăng lên 10s như khuyến nghị (Safety First)
MAX_RETRIES = 3      # Số lần thử lại tối đa nếu lỗi
INITIAL_BACKOFF = 5  # Thời gian chờ ban đầu khi lỗi (giây)

print(f"🚀 Bắt đầu chạy đánh giá! Kết quả lưu tại '{OUTPUT_FILE}'.")
print(f"ℹ️ Cấu hình an toàn: Nghỉ {SLEEP_TIME}s/câu, Backoff khởi điểm {INITIAL_BACKOFF}s.")

# --- HÀM WRAPPER: EXPONENTIAL BACKOFF (QUAN TRỌNG) ---
# Hàm này bọc lấy lệnh gọi API, nếu lỗi sẽ chờ và thử lại
async def call_with_backoff(func, *args, **kwargs):
    backoff_time = INITIAL_BACKOFF
    for attempt in range(MAX_RETRIES):
        try:
            # Gọi hàm gốc
            return await func(*args, **kwargs)
        except Exception as e:
            error_str = str(e).lower()
            # Chỉ retry nếu gặp lỗi về mạng hoặc Rate Limit (429, quota, overload)
            # Nếu lỗi logic code thì throw luôn
            is_rate_limit = "429" in error_str or "quota" in error_str or "resource exhausted" in error_str
            
            if is_rate_limit or attempt < MAX_RETRIES - 1:
                print(f"\n⚠️ Lỗi API (Lần thử {attempt + 1}/{MAX_RETRIES}). Đang chờ {backoff_time}s... Lỗi: {str(e)[:100]}...")
                await asyncio.sleep(backoff_time) # Chờ
                backoff_time *= 2 # Nhân đôi thời gian chờ (5s -> 10s -> 20s...)
            else:
                raise e # Hết số lần thử thì báo lỗi thật
    raise Exception("Max retries exceeded")

# --- MAIN LOOP ---
# if os.path.exists(OUTPUT_FILE): os.remove(OUTPUT_FILE) # Uncomment nếu muốn xóa cũ

current_batch = []

# Lưu ý: Nếu chạy trong Jupyter/Async function thì dùng range thường kết hợp tqdm
# Nếu df quá lớn, hãy cẩn thận với tqdm trong async. Ở đây mình dùng tqdm wrapper cơ bản.
for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    question = row['question']
    # print(f"Processing: {question[:50]}...") # In ít thôi cho đỡ rối log
    ground_truth = row['answer']
    row_id = row.get('id', index)

    # --- 1. CHẠY EDURAG (CÓ BACKOFF) ---
    start_rag = time.time()
    try:
        # Bọc hàm call trong call_with_backoff
        # Lưu ý: handle_query trả về tuple, ta hứng kết quả
        rag_response_text, _ = await call_with_backoff(chat_manager.handle_query, question, TOP_K)
    except Exception as e:
        error_msg = traceback.format_exc()
        print(f"❌ EDURAG thất bại ID {row_id} sau {MAX_RETRIES} lần thử:")
        # print(error_msg) # Comment lại nếu không muốn spam log
        rag_response_text = f"Error: {str(e)}"
    end_rag = time.time()
    rag_time = end_rag - start_rag

    # --- 2. CHẠY BASELINE (CÓ BACKOFF) ---
    start_base = time.time()
    try:
        # Bọc hàm call trong call_with_backoff
        base_response_text = await call_with_backoff(get_baseline_response, question)
    except Exception as e:
        print(f"❌ BASELINE thất bại ID {row_id}: {str(e)}")
        base_response_text = f"Error: {str(e)}"
    end_base = time.time()
    base_time = end_base - start_base

    # --- 3. THÊM VÀO BATCH ---
    current_batch.append({
        "id": row_id,
        "question": question,
        "ground_truth": ground_truth,
        "edurag_ans": rag_response_text,
        "edurag_time": round(rag_time, 2),
        "baseline_ans": base_response_text,
        "baseline_time": round(base_time, 2)
    })

    # --- 4. LƯU FILE THEO BATCH ---
    if len(current_batch) >= BATCH_SIZE:
        df_batch = pd.DataFrame(current_batch)
        write_header = not os.path.exists(OUTPUT_FILE)
        try:
            df_batch.to_csv(OUTPUT_FILE, mode='a', header=write_header, index=False, encoding='utf-8-sig')
            print(f"💾 Đã lưu batch {len(current_batch)} dòng.")
        except Exception as e:
            print(f"⚠️ Lỗi khi lưu file CSV: {e}")
        current_batch = [] # Reset

    # --- 5. NGHỈ NGƠI (RATE LIMITING) ---
    # Dùng await asyncio.sleep để không chặn luồng chính nếu bạn đang chạy async
    await asyncio.sleep(SLEEP_TIME) 

# --- 6. LƯU SÓT ---
if current_batch:
    df_batch = pd.DataFrame(current_batch)
    write_header = not os.path.exists(OUTPUT_FILE)
    df_batch.to_csv(OUTPUT_FILE, mode='a', header=write_header, index=False, encoding='utf-8-sig')
    print(f"💾 Đã lưu nốt {len(current_batch)} dòng cuối.")

print("\n✅ Hoàn tất đánh giá toàn bộ!")